# Corrective RAG (CRAG)

Self-RAG asks *which* retrievals are useful. **CRAG** (Yan et al., 2024) asks the harder question first: *can the retrieved set answer the question at all?*

* **confident** → answer from corpus.
* **ambiguous** → augment corpus context with a web snippet.
* **insufficient** → drop the corpus and answer from web only (or refuse).

The grader runs once per query; the web fallback is mocked here with canned snippets so the notebook stays deterministic.

In [1]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
print('LLM_CACHE_ONLY =', os.environ.get('LLM_CACHE_ONLY', '0'))


LLM_CACHE_ONLY = 1


In [2]:
from shared.embedders import cosine_topk, hash_embed
from shared.llm import Message, complete
from shared.loaders import load_corpus, load_golden_qa
from shared.prompts import RAG_SYSTEM, rag_user_prompt

MODEL = 'openai/gpt-4o-mini'
NS = '01-rag/07-corrective-rag'
DOCS = load_corpus()
QA = {q.id: q for q in load_golden_qa()}
doc_texts = [d.title + '. ' + d.abstract for d in DOCS]
doc_ids = [d.arxiv_id for d in DOCS]
doc_vecs = hash_embed(doc_texts, dims=256, seed=0)

def retrieve(q, k=3):
    qv = hash_embed([q], dims=256, seed=0)[0]
    idx, _ = cosine_topk(qv, doc_vecs, k=k)
    return [(doc_ids[i], doc_texts[i]) for i in idx]

## The grader

One short call returning a categorical verdict.

In [3]:
GRADER_SYS = (
    "You judge whether retrieved documents are sufficient to answer a question. "
    "Reply with EXACTLY one token: 'confident', 'ambiguous', or 'insufficient'."
)
def grade(q, passages):
    ctx = '\n\n'.join(f'[{d}] {t}' for d, t in passages)
    user = f'Question: {q}\n\nRetrieved:\n{ctx}\n\nVerdict:'
    out = complete(model=MODEL, namespace=NS, messages=[
        Message(role='system', content=GRADER_SYS),
        Message(role='user', content=user),
    ]).content.strip().lower()
    return out.split()[0] if out else 'insufficient'

## Mock "web" fallback

In production this is a search API. Here a tiny dict keeps the notebook deterministic.

In [4]:
WEB_SNIPPETS = {
    'q07': 'Wikipedia (Mixture-of-Experts): MoE layers route each token to a small subset of expert sub-networks.',
    'q27': '(no external source disagrees with the in-corpus refusal; topic is out of scope)',
    'q28': 'Public FAQ: this benchmark does not measure latency on edge devices.',
    'q29': 'Reddit thread: no academic paper supports the asked claim.',
    'q30': 'External survey: there is no published agent that solves this problem end-to-end.',
}
def web_search(qid: str) -> str:
    return WEB_SNIPPETS.get(qid, '(no external evidence)')

## Putting it together

In [5]:
def answer(q, contexts):
    user = rag_user_prompt(q, contexts)
    return complete(model=MODEL, namespace=NS, messages=[
        Message(role='system', content=RAG_SYSTEM),
        Message(role='user', content=user),
    ]).content

def crag(qid):
    q = QA[qid].question
    passages = retrieve(q, k=3)
    verdict = grade(q, passages)
    if verdict == 'confident':
        return verdict, answer(q, passages)
    if verdict == 'ambiguous':
        mixed = list(passages) + [('web', web_search(qid))]
        return verdict, answer(q, mixed)
    return verdict, answer(q, [('web', web_search(qid))])

for qid in ['q01', 'q11', 'q07', 'q27', 'q28']:
    v, a = crag(qid)
    print(f'--- {qid}  [{v}] ---')
    print(a)
    print()

--- q01  [confident] ---
RA-MoE has 47B total parameters with two experts active per token (~13B active).

--- q11  [confident] ---
Distill-Reason transfers CoT reasoning from large teachers to ~7B-parameter students, recovering ~80% of the gain.

--- q07  [ambiguous] ---
Polyglot-Switch adds language-conditioned routing on top of a Mixture-of-Experts backbone.

--- q27  [insufficient] ---
I cannot answer this question from the available sources.

--- q28  [insufficient] ---
Neither the corpus nor the supplemental search provides a measurement.



## What you can extend

* Replace the mock with an actual web tool (Tavily, Bing, Brave). Keep the *grader* identical.
* Add a second pass: re-query the corpus with rewritten terms before falling back to web.
* Use the verdict to *trigger* a more expensive reranker only on ambiguous cases.